# Strategy 0. Baseline knowledge

Using Extended biomix test-set to check the background knowledge of LLMs


In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [2]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

def ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")


## Questions

Loading from an extended biomix test-set

In [3]:
questions = pd.read_csv("../biomix/testset/biomix_true_false_selected_augmented.csv")

## Direct QA

In [4]:
from langchain_core.prompts import PromptTemplate

system_prompt = """
You are an expert biologist with extensive knowledge across various fields of biology, including molecular biology, genetics, medicine, and disease etiology. 
Your task is to answer a True/False biological question based on your comprehensive understanding of biological concepts and principles. 
You will be provided with a scientific question. 

Carefully analyze the statement presented in the question. Consider the following steps to answer the question:
1. Identify the key biological concepts or principles involved in the statement.
2. Recall relevant facts, theories, or experimental evidence related to these concepts.
3. Evaluate whether the statement aligns with current scientific understanding in biology and pharmacology.
4. Consider any potential exceptions or special cases that might affect the validity of the statement.

Based on your analysis, determine whether the statement is true or false. Generate your answer in the following format:

First line should contain single word: True or False 
Following that, provide explanations of why you think the statement is true or false.
"""


In [5]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 1
todo = [(m,r['text']) for _,r in questions.iterrows() for m in models for _ in range(niter)]

def run_llm_biomix00(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt}\n------------------------------------------\nUser question:\n{question}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=question)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


In [6]:
llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_biomix00(llm_model, question))


Prompting LLM: 100%|██████████| 400/400 [38:54<00:00,  5.84s/it]


In [9]:
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_biomix00(llm_model, question)
       

In [10]:
# Process answers:
# Split the answer into the first line and the rest; 
# check if True/False is in the first line. If yes - use it as the answer, otherwise - call it "Unknown"

def process_answer(answer):
    lines = answer.split("\n")
    first_line = lines[0].strip().lower()
    if "true" in first_line and "false" not in first_line:
        return {
            "answer": "True",
            "explanation": "\n".join(lines[1:]).strip()
        }
    elif "false" in first_line and "true" not in first_line:
        return {
            "answer": "False",
            "explanation": "\n".join(lines[1:]).strip()
        }
    else:
        return {
            "answer": "Unknown",
            "explanation": answer.strip()
        }

llm_answers_processed = [process_answer(a) for a in llm_answers]

In [11]:
llm_answers_df = pd.DataFrame(llm_answers_processed, columns=["answer", "explanation"])
llm_answers_df["model"] = [m for m, _ in todo]
llm_answers_df["question"] = [q for _, q in todo]
llm_answers_df = llm_answers_df.merge(questions, left_on="question", right_on="text", how="left")
llm_answers_df = llm_answers_df.drop(columns=["text"])
llm_answers_df.to_excel("biomix_00_baseline.xlsx", index=False)
llm_answers_df

,answer,explanation,model,question,label
0,False,Polycythemia Vera (PV) is indeed associated wi...,gpt-4o,Polycythemia Vera is not associated with Gene ...,False
1,False,Explanation:\n\nThis statement is false becaus...,claude-3-5-sonnet-20240620,Polycythemia Vera is not associated with Gene ...,False
2,False,Polycythemia Vera is a myeloproliferative neop...,open-mistral-7b,Polycythemia Vera is not associated with Gene ...,False
3,False,Polycythemia Vera is strongly associated with ...,o1-preview-2024-09-12,Polycythemia Vera is not associated with Gene ...,False
4,True,Cystic Fibrosis (CF) is a genetic disorder tha...,gpt-4o,Cystic Fibrosis associates Gene CFTR,True
...,...,...,...,...,...
395,False,Cystinuria is not associated with the GHR gene...,o1-preview-2024-09-12,Cystinuria is associated with Gene GHR,False
396,False,Smith-Lemli-Opitz Syndrome (SLOS) is not assoc...,gpt-4o,Smith-Lemli-Opitz Syndrome is associated with ...,False
397,False,Smith-Lemli-Opitz Syndrome (SLOS) is not assoc...,claude-3-5-sonnet-20240620,Smith-Lemli-Opitz Syndrome is associated with ...,False
398,True,Explanation:\n\nSmith-Lemli-Opitz Syndrome (SL...,open-mistral-7b,Smith-Lemli-Opitz Syndrome is associated with ...,False


In [12]:
# Calculate the fraction of correct answers for each model
accuracy_df = llm_answers_df.groupby('model').apply(lambda x: (x['label'].astype(str).str.lower() == x['answer'].str.lower()).mean()).reset_index()
accuracy_df.columns = ['model', 'accuracy']
accuracy_df

C:\Users\OStroganov\AppData\Local\Temp\ipykernel_5588\1929410681.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_df = llm_answers_df.groupby('model').apply(lambda x: (x['label'].astype(str).str.lower() == x['answer'].str.lower()).mean()).reset_index()


,model,accuracy
0,claude-3-5-sonnet-20240620,0.74
1,gpt-4o,0.93
2,o1-preview-2024-09-12,1.00
3,open-mistral-7b,0.64
